# Manual validation of the predicted mesenchymal cell states

**Developed by:** Anna Maguza\
**Affiliation:** Faculty of Medicine, Würzburg University\
**Creation date:** 31st January 2024\
**Last modified date:** 31st January 2024

This notebook outlines the process of validation of predicted by `scVI - scANVI` pipeline mesenchymal cell states annotations. We will evaluate `scANVI` classificator confidence and also validate the predicted annotation with markers.

## Import packages

In [1]:
import numpy as np
import scanpy as sc
import anndata
import pandas as pd
import plotnine as p
import matplotlib.pyplot as plt
import seaborn as sns

import json
from datetime import datetime

## Setup working environment

In [2]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi = 180, color_map = 'magma_r', dpi_save = 300, vector_friendly = True, format = 'svg')

In [3]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

In [4]:
timestamp

'31012025_153152'

## Upload data

In [5]:
adata = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_scVI_scANVI_mesenchymal_cellstates_AM_30012025_144410_raw.h5ad')
adata

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.


AnnData object with n_obs × n_vars = 185038 × 43704
    obs: 'cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparation_pro

## Validate confidence score and markers expression

In [6]:
adata_log = adata.copy()
sc.pp.normalize_total(adata_log, target_sum=1e6, exclude_highly_expressed=True)
sc.pp.log1p(adata_log)

normalizing counts per cell. The following highly-expressed genes are not considered during normalization factor computation:
['S100A6', 'ACTG2', 'MFSD1', 'APOD', 'FDCSP', 'ENSG00000286848', 'CXCL10', 'CXCL13', 'ATG10', 'CXCL14', 'GPX3', 'MLN', 'LINC01013', 'COL1A2', 'ADAMDEC1', 'CLU', 'RPL30-AS1', 'CCL19', 'CCL21', 'TPM2', 'GSN', 'VIM-AS1', 'C10orf55', 'ADIRF-AS1', 'ACTA2', 'BEST1', 'TALAM1', 'TAGLN', 'ENSG00000272173', 'MGP', 'ENSG00000273149', 'FOS', 'THBS1', 'RPLP1', 'HBA2', 'CCL2', 'CCL11', 'COL1A1', 'ENSG00000267598', 'ZFP36', 'FTL', 'TMSB4X', 'MT-RNR1', 'MT-RNR2', 'MT-CO1', 'MT-CO2']
    finished (0:00:03)


In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 15))
    sc.pl.umap(adata_log,color=["cellstates_scANVI", "confidence_score"], ncols=4, color_map = 'magma_r', frameon=False, show=False, size = 3)
    #plt.savefig(f"integration_of_remapped_data/plots/manual_annotation_validation/epithelial_scANVI_confidence_{timestamp}.png", bbox_inches="tight")
    plt.show()

## Validate markers expression

In [8]:
def visualize_cell_state_markers(adata, cell_state, markers, timestamp=None):
    """
    Create temporary annotation for specific cell state and visualize with markers.
    
    Parameters:
    -----------
    adata : AnnData
        Annotated data matrix
    cell_state : str
        Cell state of interest from cellstates_scANVI column
    markers : list
        List of marker genes to visualize
    timestamp : str, optional
        Timestamp for file naming. If None, current time will be used
    """

    if timestamp is None:
        timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

    # Create temporary column for cell state
    temp_col_name = 'temp_cell_state'
    adata.obs[temp_col_name] = adata.obs['cellstates_scANVI'] == cell_state
    
    # Set up colors in uns
    original_colors = None
    if 'temp_cell_state_colors' in adata.uns:
        original_colors = adata.uns['temp_cell_state_colors'].copy()
    
    # Set up color palette
    adata.uns[f'{temp_col_name}_colors'] = ['#D3D3D3', '#ba0f30']  # Light pink for target, light grey for others
    
    # Calculate plot layout
    n_total_plots = len(markers) + 1  # +1 for cell state plot
    n_cols = 3
    n_rows = (n_total_plots + (n_cols - 1)) // n_cols  # Ceiling division
    
    # Create the plot
    with plt.rc_context():
        sc.set_figure_params(dpi=300, figsize=(15, 5*n_rows))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
        
        # Convert axes to 1D array if necessary
        if n_rows == 1:
            axes = np.array([axes])  # Handle single row case
        axes = axes.flatten()
        
        # Plot cell state
        sc.pl.umap(adata, color=[temp_col_name], ax=axes[0], 
                  frameon=False, title=f'{cell_state} cells',
                  show=False, size=3)
        
        # Plot markers
        for i, marker in enumerate(markers):
            if i + 1 >= len(axes):
                break
            if marker in adata.var_names:
                sc.pl.umap(adata, color=marker, ax=axes[i + 1],
                          color_map='magma_r', frameon=False,
                          title=marker, show=False, size=3)
            else:
                print(f"Warning: Marker {marker} not found in the data")
                axes[i + 1].set_title(f"{marker} (not found)")
                axes[i + 1].axis('off')
        
        # Remove empty subplots
        for i in range(len(markers) + 1, len(axes)):
            fig.delaxes(axes[i])
        
        plt.tight_layout()
        # Save figure
        plt.savefig(
            f"4_mesenchymal_cell_states_annotation/figures/manual_annotation_validation/{cell_state}_markers_{timestamp}.png", 
            bbox_inches="tight"
        )
        plt.show()
    
    # Clean up: remove temporary column and colors
    del adata.obs[temp_col_name]
    if f'{temp_col_name}_colors' in adata.uns:
        del adata.uns[f'{temp_col_name}_colors']
    
    # Restore original colors if they existed
    if original_colors is not None:
        adata.uns['temp_cell_state_colors'] = original_colors

In [9]:
adata_log.obs['cellstates_scANVI'].value_counts()

cellstates_scANVI
Mesoderm           74036
Stromal            47365
SMC                19893
Myofibroblast      19226
Mesothelium        14728
Pericyte            9776
ICC                   13
Lymphoid_Stroma        1
Name: count, dtype: int64

+ Mesoderm

In [ ]:
markers = ['HAND1', 'HAND2', 'PITX2', 'ZEB2']
visualize_cell_state_markers(adata_log, "Mesoderm", markers)

+ Stromal

In [ ]:
markers = ['ADAMDEC1', 'PDGFRA','BMP4', 'C7', 'THY1', 'IGFBP4']
visualize_cell_state_markers(adata_log, "Stromal", markers)

+ SMC

In [ ]:
markers = ['DES', 'CNN1', 'ACTA2', 'MYH11']
visualize_cell_state_markers(adata_log, "SMC", markers)

+ Myofibroblast

In [ ]:
markers = ['ACTA2', 'TAGLN', 'DCN', 'DES', 'NKX3-1', 'ACTG2']
visualize_cell_state_markers(adata_log, "Myofibroblast", markers)

* Mesothelium

In [ ]:
markers = ['KRT19', 'LRRN4','UPK3B']
visualize_cell_state_markers(adata_log, "Mesothelium", markers)

* Pericyte

In [ ]:
markers = ['NOTCH3', 'MCAM', 'RGS5']
visualize_cell_state_markers(adata_log, "Pericyte", markers)

* ICC

In [ ]:
markers = ['KIT', 'ANO1', 'ETV1', 'SPON2']
visualize_cell_state_markers(adata_log, "ICC", markers)

* Lymphoid_Stroma

In [ ]:
markers = ['CCL19', 'GREM1', 'TNFSF13B', 'WFDC2', 'CXCL13', 'CR1', 'CR2', 'LMO3', 'RASD1', 'PODN']
visualize_cell_state_markers(adata_log, "Lymphoid_Stroma", markers)

+ Markers of erythroid cells

In [ ]:
sc.set_figure_params(dpi=300, figsize=(7, 7))
sc.pl.umap(adata_log,color=["HBB", "GYPA"], ncols=4, color_map = 'magma_r', frameon=False, show=False, size = 3)